# **Prediksi Biaya Klaim Medis Tahunan Berdasarkan Kondisi Klinis dan Pemanfaatan Fasilitas Kesehatan**

---
Anggota:
1. Bylbiss El Haqqie (123240003)
2. Muhammad Arif Hermawan (123240063)
3. Alifah Chairul Munawar (123240234)
4. Ikhsan Wahyu Endriarto (123240241)

# **1. Business Understanding**

## 1.1. Latar Belakang

Ketidakakuratan dalam memprediksi biaya tagihan medis tahunan nasabah menjadi ancaman serius bagi stabilitas finansial perusahaan asuransi kesehatan. Tanpa estimasi risiko medis riil yang tepat, perusahaan akan kesulitan menentukan besaran premi ideal, sehingga memicu terjadinya kesalahan penetapan harga. Penentuan premi yang terlalu rendah pada nasabah berisiko tinggi membuat perusahaan rentan menanggung pembengkakan klaim mendadak, sedangkan premi yang terlalu tinggi pada nasabah berisiko rendah mendorong mereka beralih ke kompetitor. Dampaknya, perusahaan berpotensi mengalami kerugian ganda akibat membengkaknya rasio klaim sekaligus hilangnya basis pelanggan.

## 1.2. Solusi

Sebagai solusi atas permasalahan tersebut, perusahaan perlu menerapkan model prediksi yang mampu mengestimasi biaya tagihan medis nasabah secara akurat dan objektif berdasarkan histori kondisi klinis serta pemanfaatan fasilitas kesehatan. Sehingga perusahaan dapat menetapkan tarif premi secara presisi dan adil, mencegah mispricing, serta menjaga rasio klaim tetap sehat. Langkah ini efektif mempertahankan nasabah berisiko rendah melalui harga yang kompetitif sekaligus melindungi stabilitas finansial perusahaan dari risiko pembengkakan klaim mendadak.

## 1.3. Import Library

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from pathlib import Path
import joblib

## 1.4. Memuat Dataset

In [ ]:
path_to_csv = './data/medical_insurance.csv'
df = pd.read_csv(path_to_csv)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

display(df.head())

# **2. Data Understanding**

## 2.1. Missing Values

Dataset memiliki missing value hanya ditemukan pada kolom alcohol_freq dengan presentase sebesar 30% sehingga kolom tersebut butuh ditangai dengan mengisi kategori tersebut dengan unknown pada tahap data cleaning

In [ ]:
print("Missing values:")
print(df.isna().sum())

print(f"\nTotal missing value dalam seluruh dataset: {df.isna().sum().sum()}")

Diketahui `alcohol_freq` memiliki missing values, dicari persentase missing value terhadap keseluruhan data

In [ ]:
alcohol_freq_missing_count = df['alcohol_freq'].isna().sum()
print(f"Column: alcohol_freq")
print(f"Missing count: {alcohol_freq_missing_count}")
print(f"Missing percentage: {alcohol_freq_missing_count/len(df)*100}%")

## 2.2. Data Duplikat

setelah dilakukan pemeriksaan, pada dataset tidak ditemukan data duplikat, hal ini membantu menjaga distribusi data tetep unik dan mengurangi resiko perulangan observasi.

In [ ]:
print("Data duplikat")
print(df.duplicated().sum())

## 2.3. Deteksi Outlier



### 2.3.1. Distribusi Fitur Numerik

setelah dilakukan pemeriksaan fitur numerik memiliki tingkat kemiringan yang berbeda mulai dari left-skewed, normal , hingga right-skewed dan hal tersebut dapat mempengaruhi peforma model regresi.

In [ ]:
numeric_features = df.select_dtypes(include=[np.number]).columns

skewness = df[numeric_features].skew()
normal_threshold_range = [-0.5, 0.5]
normal_features = skewness[(skewness >= normal_threshold_range[0]) & (skewness <= normal_threshold_range[1])].index.tolist()
left_skewed_features = skewness[skewness < normal_threshold_range[0]].index.tolist()
right_skewed_features = skewness[skewness > normal_threshold_range[1]].index.tolist()

print("Number of features with normal distribution:", len(normal_features))
print("Number of features with left-skewed distribution:", len(left_skewed_features))
print("Number of features with right-skewed distribution:", len(right_skewed_features))

skewness_table = pd.DataFrame({
    'Feature': skewness.index,
    'Skewness': skewness.values,
    'Distribusi': [
        'NORMAL' if normal_threshold_range[0] <= val <= normal_threshold_range[1]
        else 'RIGHT-SKEWED' if val > normal_threshold_range[1]
        else 'LEFT-SKEWED'
        for val in skewness.values
    ]
})

skewness_table['Skewness'] = skewness_table['Skewness'].round(2)

display(skewness_table)

### 2.3.2. Interquartile Range

Dengan metode QR mendeteksi outlier pada beberapa fitur numerik. nilai ekstream perlu ditangani pada tahap preparation agar tidak memberikan pengaruh berlebihan terhadap model regresi.

In [ ]:
q1 = df[numeric_features].quantile(0.25)
q3 = df[numeric_features].quantile(0.75)
iqr = q3 - q1

iqr_outliers = ((df[numeric_features] < (q1 - 1.5 * iqr)) | (df[numeric_features] > (q3 + 1.5 * iqr)))

iqr_summary = iqr_outliers.sum().reset_index()
iqr_summary.columns = ['Feature', 'Outlier Count']
iqr_summary['Percentage (%)'] = (iqr_summary['Outlier Count'] / len(df)) * 100

print("Pengecekan outlier menggunakan IQR")
display(iqr_summary)

### 2.3.3. Z-Score

penggunaan z-score mendeteksi outlier berdasarkan jarak nilai terhadap rerata dan standar deviasi. Hasil z-score lebih sensitif terhadap distribusi mendekati normal.

In [ ]:
z_score  = z_scores = np.abs(stats.zscore(df[numeric_features].fillna(df[numeric_features].median())))
z_score_outliers = pd.DataFrame(z_scores > 3, columns=numeric_features, index=df.index)

z_score_summary = pd.DataFrame({
    'Outlier Count': z_score_outliers.sum(axis=0),
    'Percentage (%)': (z_score_outliers.sum(axis=0) / len(df)) * 100
})

print("Pengecekan outlier menggunakan Z-Score")
display(z_score_summary)

In [ ]:
total_rows = len(df)
summary_results = []

for col in numeric_features:
    unique_count = df[col].nunique()

    # Deteksi tipe variabel sederhana
    if unique_count <= 2:
        data_type = "Binary"
    elif unique_count < 10 and (df[col] % 1 == 0).all():
        data_type = "Nominal/Ordinal"
    else:
        data_type = "Kontinu"

    iqr_outlier_count = iqr_outliers[col].sum()
    z_score_outlier_count = z_score_outliers[col].sum()

    summary_results.append({
        'Nama Kolom': col,
        'Tipe Data': data_type,
        'Total Outlier (IQR)': iqr_outlier_count,
        '% Outlier (IQR)': round((iqr_outlier_count / total_rows) * 100, 2),
        'Total Outlier (Z-Score)': z_score_outlier_count,
        '% Outlier (Z-Score)': round((z_score_outlier_count / total_rows) * 100, 2)
    })

summary_table = pd.DataFrame(summary_results)

# hitung Total Baris Unik yang Terkena Outlier
total_iqr_affected_rows = iqr_outliers.any(axis=1).sum()
total_zscore_affected_rows = z_score_outliers.any(axis=1).sum()

print(f"=== RINGKASAN TOTAL BARIS TERDAMPAK ===")
print(f"Total Baris Data: {total_rows}")
print(f"Baris dengan minimal 1 Outlier (IQR)    : {total_iqr_affected_rows} ({total_iqr_affected_rows / total_rows * 100:.2f}%)")
print(f"Baris dengan minimal 1 Outlier (Z-Score): {total_zscore_affected_rows} ({total_zscore_affected_rows / total_rows * 100:.2f}%)\n")



### 2.3.4. Visualisasi Outlier
Visualisasi distribusi data menggunakan boxplot untuk mendeteksi outlier pada setiap fitur numerik. Boxplot memperlihatkan titik ekstream pada beberapa fitur yang  mendukung temuan metode IQR dan Z-Score, sehingga capping pada nilai ekstream untuk menjaga jumlah observasi.

In [ ]:
fig, axes = plt.subplots(
    nrows=6, 
    ncols=5, 
    figsize=(20, 20)
)

for ax, feature in zip(axes.flatten(), numeric_features):
    sns.boxplot(
        data=df,
        x=feature,
        ax=ax,
        orient='h'
    )

plt.tight_layout()
plt.show()

## 2.4. Exploratory Data Analysis

### 2.4.1. Analisis Distribusi Target

fitur annual_medical_cost memiliki distribusi yang cenderung right-skwed, jika nilai target ekstream dapat mempengaruhi estimasi model, sehingga perlu diperhatikan pada tahap preparation dan evaluation.

In [ ]:
# Visualisasi distribusi target 'annual_medical_cost'
plt.figure(figsize=(12, 6))
sns.histplot(df['annual_medical_cost'], kde=True, bins=50, color='teal')
plt.title('Distribusi Biaya Medis Tahunan (Annual Medical Cost)', fontsize=16)
plt.xlabel('Annual Medical Cost (USD)', fontsize=12)
plt.ylabel('Frekuensi', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("\nStatistik Deskriptif untuk 'annual_medical_cost':")
print(df['annual_medical_cost'].describe())

### 2.4.2. Distribusi Fitur Numerik Klinis

Fitur numerik klinis memiliki nilai rata-rata, median, dan rentang yang berbeda-beda. Perbedaan skala tersebut membuat proses scaling diperlukan agar model tidak terlalu dipengaruhi oleh fitur bernilai besar.

In [ ]:

numerical_features = [
    'age', 'bmi', 'systolic_bp', 'diastolic_bp', 'ldl', 'hba1c',
    'visits_last_year', 'medication_count', 'chronic_count', 'hospitalizations_last_3yrs',
    'days_hospitalized_last_3yrs', 'proc_imaging_count', 'proc_surgery_count', 'proc_physio_count',
    'proc_consult_count', 'proc_lab_count'
]

summary_num = df[numerical_features].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
summary_num = summary_num.round(2).reset_index().rename(columns={'index': 'Feature'})
display(summary_num)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

summary_num.set_index('Feature')['mean'].sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Rata-rata (Mean) per Fitur Numerik', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Mean')
axes[0].grid(axis='x', linestyle='--', alpha=0.7)

summary_num.set_index('Feature')[['mean', '50%']].sort_values('mean').plot(
    kind='barh', ax=axes[1], color=['steelblue', 'orange'], edgecolor='black'
)
axes[1].set_title('Mean vs Median per Fitur Numerik', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Nilai')
axes[1].legend(['Mean', 'Median'])
axes[1].grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


### 2.4.3. Distribusi Fitur Kategorikal

Distribusi kategorikal tidak selalu seimbang karena karena beberapa kategori dan kondisi nilai yang paling sering muncul (modus). ketidakseimbangan ini perlu diperhatikan karena karena dapat mempengaruhi kemampuan model mengenali pola data.

In [ ]:
categorical_features = [
    'sex', 'smoker', 'alcohol_freq', 'hypertension', 'diabetes',
    'copd', 'cardiovascular_disease', 'cancer_history',
    'kidney_disease', 'liver_disease', 'arthritis', 'mental_health', 'asthma'
]

rows = []
for f in categorical_features:
    vc = df[f].value_counts(dropna=False)
    pct = df[f].value_counts(normalize=True, dropna=False) * 100
    for val, cnt in vc.items():
        label = 'Missing' if pd.isna(val) else str(val)
        rows.append({
            'Feature': f,
            'Kategori': label,
            'Jumlah': int(cnt),
            'Persentase (%)': round(float(pct[val]), 2)
        })

cat_summary = pd.DataFrame(rows)
display(cat_summary)

n_cols = 3
n_rows = int(np.ceil(len(categorical_features) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, f in enumerate(categorical_features):
    vc = df[f].value_counts(dropna=False)
    labels = ['Missing' if pd.isna(x) else str(x) for x in vc.index]
    sns.barplot(x=labels, y=vc.values, ax=axes[i], hue=labels, palette='viridis', legend=False)
    axes[i].set_title(f'Distribusi "{f}"', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Jumlah')
    axes[i].tick_params(axis='x', rotation=30)
    # Anotasi nilai
    for j, v in enumerate(vc.values):
        axes[i].text(j, v, f'{v:,}', ha='center', va='bottom', fontsize=8)

for k in range(len(categorical_features), len(axes)):
    axes[k].axis('off')

plt.tight_layout()
plt.show()

### 2.4.4. Analisis Korelasi Fitur dengan Target

Tidak semua fitur numerik memiliki kolerasi linear yang kuat terhadap annual_medical_cost, dan fitur dengan kolerasi tertinggi dapat menjadi prediktor tanpa menunjukkan hubungan sebab akibat

In [ ]:
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(24, 22))
sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm', 
    fmt=".2f", 
    linewidths=.5, 
    annot_kws={"size": 8},
    mask=np.triu(np.ones_like(corr_matrix, dtype=bool))
)
plt.title('Heatmap Korelasi Antar Fitur dan dengan Target', fontsize=20)
plt.show()

print("\nKorelasi Fitur dengan 'annual_medical_cost' (diurutkan):")
print(corr_matrix['annual_medical_cost'].abs().sort_values(ascending=False))

### 2.4.5. Analisis Korelasi Prediktor dengan Target

Usia dan BMI menunjukkan hubungan yang tidak sepenuhnya linear dengan biaya medis tahunan. Sementara itu, visits_last_year dan medication_count memperlihatkan hubungan yang kompleks. Oleh karena itu, karakter non-linear dan interaksi antar fitur perlu dipertimbangkan dalam pemodelan.

In [ ]:
xs = ["age", "visits_last_year", "mental_health", "smoker", "bmi", "medication_count"]

fig, axes = plt.subplots(
    nrows=3, 
    ncols=2,    
    figsize=(12, 10)
)

for ax, feature in zip(axes.flatten(), xs):
    sns.scatterplot(
        data=df,
        x=feature,
        y='annual_medical_cost',
        ax=ax,
        alpha=0.6
    )
    ax.set_xlabel(feature, fontsize=12)
    ax.set_ylabel('Annual Medical Cost', fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout(pad=2.0, h_pad=2.0, w_pad=2.0)
plt.show()

Semakin bertambah usia, semakin tinggi biaya tagihan medis tahunan, puncaknya terjadi pada usia 50-60 tahun. Namun terjadi penurunan biaya medis setelah usia 60-70 tahun. Ini menunjukkan bahwa hubungan keduanya **tidak benar-benar linear**. Kasus yang sama untuk bmi.

Mental health menunjukkan distribusi yang seimbang antara terdiagnosis dan tidak terdiagnosis, sehingga hubungannya dengan biaya tagihan medis tahunan menunjukkan **korelasi positif yang lemah**.

Visits last year dan medication count menunjukkan distribusi yang mirip, dimana semakin sedikit nilai keduanya, semakin tinggi biaya tagihan medis tahunan. Hal ini menunjukkan bahwa semakin sering seseorang mengunjungi fasilitas kesehatan dan semakin banyak obat yang dikonsumsi, belum tentu biaya yang dikeluarkan semakin tinggi.

In [ ]:
x = df["visits_last_year"]
y = df["chronic_count"]
z = df["annual_medical_cost"]

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(x, y, c=z, cmap='viridis', s=100, alpha=0.8)
cbar = plt.colorbar(scatter)
cbar.set_label('Annual Medical Cost', rotation=270, labelpad=20)

ax.set_xlabel('Visits Last Year', fontsize=12)
ax.set_ylabel('Chronic Count', fontsize=12)

plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.scatterplot(
    data=df,
    x='visits_last_year',
    y='days_hospitalized_last_3yrs',
    hue='annual_medical_cost',
)

# **3. Data Preparation**

Pada tahap ini, data disiapkan agar dapat digunakan oleh model. Prosesnya meliputi menangani nilai yang belum diketahui dan nilai ekstrem, menentukan fitur dan target, membagi data menjadi data latih dan data uji, serta mengubah fitur agar sesuai dengan kebutuhan model.

## 3.1. Data Cleaning

Data asli disalin terlebih dahulu agar tetap tersedia sebagai pembanding. Nilai kosong pada kolom `alcohol_freq` diisi dengan kategori `Unknown`, sehingga baris tersebut tetap dapat digunakan tanpa menganggap frekuensi konsumsi alkoholnya sebagai nilai tertentu.

Nilai ekstrem pada sejumlah kolom numerik dibatasi menggunakan metode Interquartile Range (IQR). Nilai di bawah batas bawah diganti dengan batas bawah, sedangkan nilai di atas batas atas diganti dengan batas atas. Baris data tidak dihapus, tetapi nilai ekstremnya dibatasi agar pengaruhnya terhadap analisis berkurang.

In [ ]:
# missing value
df_cleaned = df.copy()

# Imputasi kolom alcohol_freq dengan kategori 'Unknown'
df_cleaned['alcohol_freq'] = df_cleaned['alcohol_freq'].fillna('Unknown')

print("=== VERIFIKASI MISSING VALUE ===")
print("Jumlah NaN alcohol_freq Sebelum :", df['alcohol_freq'].isnull().sum())
print("Jumlah NaN alcohol_freq Sesudah :", df_cleaned['alcohol_freq'].isnull().sum())
print("-" * 50)

# outlier
continuous_cols = [
    'income', 'bmi', 'systolic_bp', 'diastolic_bp', 
    'ldl', 'hba1c', 'annual_medical_cost', 'annual_premium', 
    'monthly_premium', 'avg_claim_amount', 'total_claims_paid'
]

# Capping dengan batas IQR pada df_cleaned
for col in continuous_cols:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Batasi nilai ekstrem tanpa menghapus baris
    df_cleaned[col] = df_cleaned[col].clip(lower=lower_bound, upper=upper_bound)

print("Capping Outlier Selesai untuk", len(continuous_cols), "kolom numerik!")

### 3.1.1. Visualisasi sebelum vs sesudah

Boxplot digunakan untuk membandingkan distribusi setiap kolom sebelum dan sesudah proses pembersihan. Perbandingan ini membantu melihat apakah nilai ekstrem berhasil dibatasi dan apakah data tetap tersedia setelah proses tersebut. Pada proses capping, baris tidak dibuang; hanya nilai yang melewati batas IQR yang disesuaikan.

In [ ]:
# Visualisasi Komparasi Sebelum vs Sesudah Cleaning (Contoh pada annual_medical_cost & bmi)
cols_to_plot = ['annual_medical_cost', 'bmi']

for col in cols_to_plot:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Sebelum
    sns.boxplot(data=df, x=col, ax=axes[0], color='salmon')
    axes[0].set_title(f'SEBELUM Cleaning: {col}')
    
    # Sesudah
    sns.boxplot(data=df_cleaned, x=col, ax=axes[1], color='skyblue')
    axes[1].set_title(f'SESUDAH Cleaning (Capped): {col}')
    
    plt.tight_layout()
    plt.show()

## 3.2. Penanganan Inkonsistensi Data

### 3.2.1. Filter Usia Minimum 18 th
 umumnya anak-anak tidak dianggap sebagai pemegang asuransi di Amerika Serikat, namun sebagai dependant. Author menyarankan untuk menerapkan filter >= 18 tahun. Sehingga data yang dipakai adalah data individu yang berusia 18 tahun ke atas.

In [ ]:
print("Verifikasi filter usia:")
print(f"Jumlah data dengan usia >= 18 tahun: {(df_cleaned['age'] >= 18).sum()}")
print(f"Jumlah data dalam df saat ini: {len(df_cleaned)}")
print(f"Nilai usia minimum pada df: {df_cleaned['age'].min()}")
print(f"Nilai usia maksimum pada df: {df_cleaned['age'].max()}")

rows_before = len(df_cleaned)
df_cleaned = df_cleaned.loc[df_cleaned["age"] >= 18].copy()
df_cleaned = df_cleaned.reset_index(drop=True)

rows_removed = rows_before - len(df_cleaned)

print(f"Jumlah data awal: {rows_before}")
print(f"Jumlah data usia >= 18 tahun: {len(df_cleaned)}")
print(f"Jumlah data yang dihapus: {rows_removed}")
print(f"Persentase data yang dihapus: {rows_removed / rows_before * 100:.2f}%")

### 3.2.2. Penanganan Inkonsistensi Age vs Marital Status
Ditemukan anomali data di mana terdapat individu berusia di bawah 18 tahun yang berstatus menikah. Hal ini tidak masuk akal secara logika dan hukum, karena usia minimum legal untuk menikah di sebagian besar negara bagian Amerika Serikat adalah 18 tahun. Selain itu, perusahaan asuransi memerlukan legalitas pernikahan untuk menimbang status tanggungan. 

In [ ]:
print("Sebelum Perbaikan")
underage_married = df.query("age < 18 and marital_status == 'Married'").shape[0]
underage_10_married = df.query("age < 10 and marital_status == 'Married'").shape[0]
print(f"Individu < 18 tahun berstatus 'Married': {underage_married}")
print(f"Individu < 10 tahun berstatus 'Married': {underage_10_married}")

In [ ]:
print("Setelah Perbaikan")
df.loc[(df['age'] < 18) & (df['marital_status'] == 'Married'), 'marital_status'] = 'Single'
underage_married = df.query("age < 18 and marital_status == 'Married'").shape[0]
print(f"Individu < 18 tahun berstatus 'Married': {underage_married}")

### 3.2.3. Penanganan Inkonsistensi Age vs Education
Ditemukan 1,2% data individual yang tidak sesuai dengan usia minimum untuk menempuh (lulus) pendidikan di berbagai tingkatan.

In [ ]:
print("Banyaknya:")
print(f"Individu yang lulus HS (High School) kurang dari 13 tahun: {df.query("education == 'HS' and age < 13").value_counts().sum()}")
print(f"Individu yang lulus Bachelors kurang dari 15 tahun: {df.query("education == 'Bachelors' and age < 15").value_counts().sum()}")
print(f"Individu yang lulus Masters kurang dari 16 tahun: {df.query("education == 'Masters' and age < 16").value_counts().sum()}")
print(f"Individu yang lulus Doctorate kurang dari 20 tahun: {df.query("education == 'Doctorate' and age < 20").value_counts().sum()}")

In [ ]:
df_cleaned = df_cleaned.drop(df_cleaned.query("education == 'HS' and age < 13").index)
df_cleaned = df_cleaned.drop(df_cleaned.query("education == 'Bachelors' and age < 15").index)
df_cleaned = df_cleaned.drop(df_cleaned.query("education == 'Masters' and age < 16").index)
df_cleaned = df_cleaned.drop(df_cleaned.query("education == 'Doctorate' and age < 20").index)

print("Jumlah data setelah penanganan inkonsistensi Age vs Education:", len(df_cleaned))

### 3.2.4. Penanganan Inkonsistensi Age vs Employment Status
Ditemukan 865 (0,82%) data individual yang memiliki usia di bawah 18 tahun tetapi status pekerjaan mereka adalah "Employed" atau "Retired". Sesuai dengan ketentuan umum di Amerika Serikat Fair Labor Standards Act of 1938 (FLSA) under Title 29 of the U.S. Code.


In [ ]:
print(df.query("age < 14 and employment_status in ['Employed', 'Retired']").value_counts().sum())

## 3.3. Feature Selection dan Data Splitting

Pada tahap ini ditentukan fitur yang digunakan sebagai masukan model (`X`) dan nilai yang ingin diprediksi sebagai target (`y`). Fitur mencakup informasi demografis, kondisi kesehatan, serta pemanfaatan layanan kesehatan. Target yang diprediksi adalah `annual_medical_cost`.

Data kemudian dibagi menjadi 80% data latih dan 20% data uji. Data latih digunakan untuk membangun model, sedangkan data uji disiapkan untuk menilai prediksi pada data yang tidak digunakan saat pelatihan. `random_state=42` membuat pembagian data dapat diulang dengan hasil yang sama.

In [ ]:
selected_features = [
    'age', 'sex', 'bmi', 'smoker', 'alcohol_freq',
    'hypertension', 'diabetes', 'copd', 'cardiovascular_disease',
    'cancer_history', 'kidney_disease', 'liver_disease', 'arthritis',
    'mental_health', 'asthma', 'chronic_count', 'systolic_bp',
    'diastolic_bp', 'ldl', 'hba1c', 'visits_last_year',
    'hospitalizations_last_3yrs', 'days_hospitalized_last_3yrs',
    'medication_count', 'proc_imaging_count', 'proc_surgery_count',
    'proc_physio_count', 'proc_consult_count', 'proc_lab_count'
]

X = df_cleaned[selected_features]
y = df_cleaned['annual_medical_cost']

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42
)

print(f"Jumlah data latih: {X_train.shape}")
print(f"Jumlah data uji: {X_test.shape}")

## 3.4. Data Preprocessing

Sebelum digunakan oleh model, fitur perlu diubah ke format yang sesuai. Fitur kategorikal diubah menjadi kolom numerik, sementara fitur numerik diskalakan. Data uji diproses dengan struktur kolom dan parameter yang sama seperti data latih agar keduanya dapat digunakan secara konsisten.

In [ ]:
categorical_selected_cols = ["sex", "smoker", "alcohol_freq"]
numerical_selected_cols = [col for col in selected_features if col not in categorical_selected_cols]

### 3.4.1 Encoding

Kolom `sex`, `smoker`, dan `alcohol_freq` merupakan fitur kategorikal, sehingga diubah menjadi kolom indikator bernilai 0 atau 1 menggunakan one-hot encoding. Opsi `drop_first=True` menghilangkan satu kategori acuan dari setiap fitur untuk mengurangi kolom yang berulang.

Kolom pada data uji kemudian disesuaikan agar sama dengan kolom data latih. Jika suatu kategori muncul di data latih tetapi tidak muncul di data uji, kolom tersebut tetap dibuat dengan nilai 0 pada data uji.

In [ ]:
X_train_encoded = pd.get_dummies(
    X_train, 
    columns=categorical_selected_cols, 
    drop_first=True,
    dtype=int
)

X_test_encoded = pd.get_dummies(
    X_test, 
    columns=categorical_selected_cols, 
    drop_first=True,
    dtype=int
)

X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns, 
    fill_value=0
)

### 3.4.2. Scaling

StandardScaler digunakan untuk menyetarakan skala fitur numerik. Parameter scaling dihitung dari data latih, lalu digunakan untuk mengubah data latih dan data uji. Fitur kategorikal hasil encoding tetap bernilai 0 atau 1, dan kolom target tidak diskalakan. Data yang telah diproses ini digunakan sebagai masukan model pada tahap modeling.

In [ ]:
scaler = StandardScaler()

X_train_encoded[numerical_selected_cols] = scaler.fit_transform(X_train_encoded[numerical_selected_cols])

X_test_encoded[numerical_selected_cols] = scaler.transform(X_test_encoded[numerical_selected_cols])

# **4. Modeling**

Pada tahap ini, beberapa model regresi dilatih untuk memprediksi biaya medis tahunan. Model menggunakan fitur yang telah melalui encoding dan scaling pada tahap Data Preparation. Kinerja model dibandingkan berdasarkan hasil prediksi pada data uji.

### 4.1 Pelatihan Model

Linear Regression digunakan sebagai model dasar. Ridge dan Lasso juga dicoba karena menerapkan regularisasi untuk mengendalikan koefisien model. Ketiga model dilatih menggunakan data latih yang telah diproses pada bagian 3.3, kemudian digunakan untuk memprediksi biaya medis pada data uji.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0, max_iter=10000),
}

predictions = {}
results = []

for name, model in models.items():
    # X_train_encoded dan X_test_encoded sudah melalui scaling di bagian 3.3.2
    model.fit(X_train_encoded, y_train)

    y_pred = model.predict(X_test_encoded)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    predictions[name] = y_pred

    results.append({
        "Model": name,
        "RMSE": rmse,
        "R2": r2,
    })

### 4.2 Perbandingan Model

Hasil prediksi dibandingkan menggunakan RMSE dan R². RMSE mengukur besarnya kesalahan prediksi dalam satuan biaya medis; semakin kecil nilainya, semakin kecil kesalahannya. R² menunjukkan seberapa baik model menjelaskan variasi target; semakin tinggi nilainya, semakin baik. Tabel diurutkan berdasarkan RMSE, sehingga model pada urutan pertama memiliki RMSE terendah di antara model yang dibandingkan.

In [ ]:
results_df = (
    pd.DataFrame(results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(results_df)

best_model_name = results_df.loc[0, "Model"]
print(f"Model dengan RMSE terendah: {best_model_name}")

# **5. Evaluation**

### 5.1 Metrik Evaluasi

Pada tahap evaluasi digunakan beberapa metrik untuk menilai kemampuan model dalam memprediksi biaya klaim medis tahunan:

- **Mean Absolute Error (MAE)** menghitung rata-rata selisih mutlak antara nilai aktual dan nilai prediksi. Semakin kecil MAE, semakin akurat prediksi model.
- **Root Mean Squared Error (RMSE)** menghitung akar dari rata-rata kuadrat kesalahan prediksi. RMSE memberikan penalti lebih besar terhadap kesalahan prediksi yang besar.
- **R-squared ($R^2$)** menunjukkan proporsi variasi biaya medis tahunan yang dapat dijelaskan oleh model. Nilai $R^2$ mendekati 1 menunjukkan performa model yang semakin baik.

Pada evaluasi ini, MAE dan RMSE dinyatakan dalam satuan biaya medis, sedangkan $R^2$ dinyatakan dalam rentang antara 0 hingga 1.

### 5.2 Evaluasi Model pada Data Uji

Evaluasi model dilakukan pada data uji untuk mengukur kemampuan generalisasi model terhadap data yang belum pernah digunakan saat pelatihan. Semakin kecil MAE dan RMSE serta semakin tinggi $R^2$, semakin baik performa model dalam memprediksi biaya medis tahunan.

In [ ]:
evaluation_results = []

for name, y_pred in predictions.items():
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    evaluation_results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })

evaluation_df = pd.DataFrame(evaluation_results).sort_values('RMSE')

display(evaluation_df.round({'MAE': 2, 'RMSE': 2, 'R2': 4}))

### 5.3 Visualisasi Aktual vs Prediksi

Garis merah putus-putus menunjukkan prediksi sempurna, yaitu kondisi ketika nilai prediksi sama dengan nilai aktual. Titik yang berada lebih dekat dengan garis tersebut memiliki kesalahan prediksi yang lebih kecil. Penyebaran titik yang jauh dari garis menunjukkan adanya kesalahan prediksi pada beberapa data uji.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, y_pred, alpha=0.5)
    ax.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        'r--',
        linewidth=2,
        label='Prediksi Sempurna'
    )
    ax.set_title(name)
    ax.set_xlabel('Biaya Medis Aktual')
    ax.set_ylabel('Biaya Medis Prediksi')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.4 Analisis Residual

Residual dihitung sebagai selisih antara nilai aktual dan nilai prediksi:

$$Residual = Nilai\ Aktual - Nilai\ Prediksi$$

Residual yang tersebar acak di sekitar garis nol menunjukkan bahwa model belum memiliki pola kesalahan sistematis yang mencolok. Sebaliknya, apabila residual membentuk pola tertentu, seperti lengkungan atau penyebaran yang semakin melebar, hal tersebut dapat menunjukkan bahwa model linear belum sepenuhnya sesuai atau masih terdapat hubungan non-linear yang belum ditangkap.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    residuals = y_test - y_pred

    ax.scatter(y_pred, residuals, alpha=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax.set_title(f'Residual - {name}')
    ax.set_xlabel('Biaya Medis Prediksi')
    ax.set_ylabel('Residual')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.5 Pemilihan Model Terbaik

Pemilihan model terbaik dilakukan berdasarkan kombinasi nilai MAE, RMSE, dan $R^2$ pada data uji. Model dengan RMSE terendah dipilih sebagai kandidat terbaik karena mampu menghasilkan kesalahan kuadrat rata-rata yang paling kecil. Namun, hasil tersebut juga perlu ditinjau bersama MAE dan $R^2$ agar keputusan tidak hanya bergantung pada satu metrik.

Berdasarkan tabel evaluasi, model terbaik ditetapkan sebagai model yang memiliki RMSE terendah dan performa yang paling konsisten pada ketiga metrik. Model tersebut selanjutnya dapat dipertimbangkan sebagai model akhir untuk digunakan pada tahap deployment.

In [ ]:
best_model_name = evaluation_df.iloc[0]['Model']
best_row = evaluation_df.iloc[0]
best_model = models[best_model_name]

print(f"Model terbaik berdasarkan RMSE terendah: {best_model_name}")
print(f"MAE  : {best_row['MAE']:.2f}")
print(f"RMSE : {best_row['RMSE']:.2f}")
print(f"R2   : {best_row['R2']:.4f}")

### 5.6 Konklusi

Berdasarkan hasil evaluasi pada data uji, model **Ridge** dipilih sebagai model terbaik karena menghasilkan **RMSE terendah**, yaitu **1828.32**, sekaligus MAE sebesar **1432.71** dan nilai $R^2$ sebesar **0.1974**. Perbedaan kinerja antara Ridge, Linear Regression, dan Lasso sangat kecil, yang menunjukkan bahwa ketiga model memiliki kemampuan prediktif yang relatif setara.

Secara keseluruhan, ketiga model baru mampu menjelaskan sekitar **19.74% variasi biaya klaim medis tahunan**. Oleh karena itu, performa model masih perlu ditingkatkan melalui penambahan fitur yang relevan, rekayasa fitur yang lebih baik, atau eksplorasi algoritma lain. Model Ridge dipilih sebagai model akhir untuk tahap deployment karena memberikan keseimbangan terbaik antara kesederhanaan, kemampuan generalisasi, dan kesalahan prediksi terendah pada data uji.

# **6. Deployment**


In [ ]:
joblib.dump({
    "model": best_model,
    "scaler": scaler,
    "columns": X_train_encoded.columns.tolist(),
    "cat_cols": categorical_selected_cols,
    "num_cols": numerical_selected_cols
}, "./models/model.joblib")
